# 🍽️ Weekly Themed Meal Planner MVP

A meal planning assistant that generates 7 themed dinners with consolidated shopping lists.

## Features
- 7 unique themed dinners (Monday-Sunday)
- 5 ingredients per recipe (excluding basics)
- Dietary preference support (vegan, vegetarian, keto, omnivore)
- Consolidated shopping list
- Email delivery

## 1. Installation & Setup

In [1]:
%pip install -q langchain langchain-openai langchain-anthropic pydantic gradio python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

Create a file called `.env` in the same folder as this notebook with this line:

```
OPENAI_API_KEY=sk-...
```

**Edit the values below before running the notebook.**

In [2]:
# ============================================================
# CONFIGURATION - Edit these values
# ============================================================
import os
from dotenv import load_dotenv

load_dotenv(".env")

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Model settings
OPENAI_MODEL = "gpt-4o-mini"
TEMPERATURE = 0.7

# Email settings (Gmail with App Password)
EMAIL_SENDER = "provide your email"
EMAIL_PASSWORD = "password"

# Meal generation settings
INGREDIENTS_PER_RECIPE = 5
EXCLUDED_BASICS = ["salt", "pepper", "oil", "garlic", "water"]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

In [3]:
# Set environment variables
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

## 3. Pydantic Models for Validation

In [4]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum


class DietaryPreference(str, Enum):
    OMNIVORE = "omnivore"
    VEGETARIAN = "vegetarian"
    VEGAN = "vegan"
    KETO = "keto"


class Ingredient(BaseModel):
    """Single ingredient with quantity."""
    name: str = Field(..., description="Name of the ingredient")
    quantity: str = Field(..., description="Quantity with unit (e.g., '2 cups', '500g')")


class Recipe(BaseModel):
    """A single themed dinner recipe."""
    day: str = Field(..., description="Day of the week")
    theme: str = Field(..., description="Cuisine theme (e.g., 'Italian', 'Mexican')")
    dish_name: str = Field(..., description="Name of the dish")
    ingredients: List[Ingredient] = Field(..., description="List of 5 ingredients")
    instructions: str = Field(..., description="Brief cooking instructions")

    @field_validator('ingredients')
    @classmethod
    def validate_ingredient_count(cls, v):
        if len(v) != INGREDIENTS_PER_RECIPE:
            raise ValueError(f"Recipe must have exactly {INGREDIENTS_PER_RECIPE} ingredients, got {len(v)}")
        return v


class ShoppingItem(BaseModel):
    """Consolidated shopping list item."""
    name: str = Field(..., description="Ingredient name")
    total_quantity: str = Field(..., description="Total quantity needed")
    used_in: List[str] = Field(..., description="List of dishes using this ingredient")


class MealPlan(BaseModel):
    """Complete weekly meal plan."""
    dietary_preference: str = Field(..., description="Selected dietary preference")
    recipes: List[Recipe] = Field(..., description="List of 7 recipes for the week")
    shopping_list: List[ShoppingItem] = Field(..., description="Consolidated shopping list")

    @field_validator('recipes')
    @classmethod
    def validate_recipe_count(cls, v):
        if len(v) != 7:
            raise ValueError(f"Meal plan must have exactly 7 recipes, got {len(v)}")
        return v

    @field_validator('recipes')
    @classmethod
    def validate_unique_themes(cls, v):
        themes = [r.theme.lower() for r in v]
        if len(themes) != len(set(themes)):
            raise ValueError("All 7 themes must be unique")
        return v

## 4. LangChain Setup & Prompts

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


def get_llm():
    """Get the configured LLM."""
    return ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE)


# Main meal plan generation prompt
MEAL_PLAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a professional meal planner. Generate a weekly meal plan with EXACTLY 7 themed dinners.

STRICT RULES - FOLLOW EXACTLY:
1. Each recipe MUST have EXACTLY 5 main ingredients (do NOT count salt, pepper, oil, garlic, or water)
2. ALL 7 THEMES MUST BE COMPLETELY DIFFERENT - no duplicates allowed! Use: Italian, Mexican, Japanese, Indian, Thai, Greek, American, Chinese, French, Korean, Mediterranean, Middle Eastern, etc.
3. All ingredients MUST be real, commonly available items
4. Follow the dietary preference strictly
5. Provide realistic quantities for 2 servings

Dietary Guidelines:
- Omnivore: Any ingredients allowed
- Vegetarian: No meat or fish, eggs and dairy OK
- Vegan: No animal products at all
- Keto: Low carb, high fat, no grains/sugar/starchy vegetables

Output valid JSON matching this exact structure:
{format_instructions}"""),
    ("human", """Create a weekly meal plan:
- Diet: {dietary_preference}
- Theme requests: {theme_requests}
- Preferences: {additional_preferences}

IMPORTANT: Each day MUST have a DIFFERENT cuisine theme. Generate 7 unique themed dinners for Monday-Sunday.""")
])


# Shopping list consolidation prompt
SHOPPING_LIST_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a shopping list organizer. Consolidate ingredients from multiple recipes into a single shopping list.

RULES:
1. Combine same ingredients and sum their quantities
2. List which dishes use each ingredient
3. Use standard units (cups, tablespoons, grams, pieces)
4. Round up quantities for convenience

Output valid JSON matching this structure:
{format_instructions}"""),
    ("human", """Consolidate the shopping list from these recipes:
{recipes_json}""")
])

## 5. Meal Generation Logic

In [25]:
import json
import time


class MealPlanGenerator:
    def __init__(self):
        self.llm = get_llm()
        self.recipe_parser = JsonOutputParser(
            pydantic_object=MealPlan
        )

    def _build_request(
        self,
        dietary_preference,
        theme_requests,
        additional_preferences
    ):
        return {
            "dietary_preference": dietary_preference,
            "theme_requests": theme_requests or "random diverse cuisines",
            "additional_preferences": additional_preferences or "none",
            "format_instructions": (
                self.recipe_parser.get_format_instructions()
            ),
        }

    def _parse_response(self, content):
        if "```json" in content:
            content = content.split("```json", 1)[1]
            content = content.split("```", 1)[0]
        elif "```" in content:
            content = content.split("```", 1)[1]
            content = content.split("```", 1)[0]

        return MealPlan(**json.loads(content.strip()))

    def _success_response(self, meal_plan, start_time):
        return {
            "success": True,
            "meal_plan": meal_plan.model_dump(),
            "generation_time": round(time.time() - start_time, 2),
            "error": None,
        }

    def _error_response(self, error):
        return {
            "success": False,
            "meal_plan": None,
            "generation_time": None,
            "error": str(error),
        }

    def generate(
    self,
    dietary_preference: str,
    theme_requests: str = "random",
    additional_preferences: str = "none",
) -> dict:
        """Generate a complete meal plan with shopping list."""
        start_time = time.time()

        try:
            result = (MEAL_PLAN_PROMPT | self.llm).invoke(
                self._build_request(
                    dietary_preference,
                    theme_requests,
                    additional_preferences,
            )
        )

            meal_plan = self._parse_response(result.content)
            return self._success_response(meal_plan, start_time)

        except Exception as error:
            return {
                "success": False,
                "meal_plan": None,
                "generation_time": None,
                "error": f"{type(error).__name__}: {error}",
        }


# Initialize generator
generator = MealPlanGenerator()

## 6. Email Functionality

In [28]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

def format_ingredients(ingredients: list[dict]) -> str:
    """Format recipe ingredients as a single readable line."""
    return ", ".join(
        f"{ingredient['quantity']} {ingredient['name']}"
        for ingredient in ingredients
    )

def format_meal_plan_email(meal_plan: dict) -> str:
    """Format meal plan as readable email content (shopping list first)."""
    lines = []
    lines.append("=" * 40)
    lines.append(f"WEEKLY MEAL PLAN ({meal_plan['dietary_preference'].upper()})")
    lines.append("=" * 40)
    lines.append("")

    # Shopping list FIRST
    lines.append("SHOPPING LIST")
    lines.append("-" * 20)
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} {item['name']}")
    lines.append("")

    # Recipes
    lines.append("=" * 40)
    lines.append("RECIPES")
    lines.append("=" * 40)
    for recipe in meal_plan['recipes']:
        lines.append(f"\n{recipe['day'].upper()} - {recipe['theme']}")
        lines.append(f"{recipe['dish_name']}")
        ingredients = format_ingredients(recipe["ingredients"])
        lines.append(f"Ingredients: {ingredients}")
        lines.append(f"Ingredients: {ingredients}")
        lines.append(f"Instructions: {recipe['instructions']}")

    return "\n".join(lines)


def build_meal_plan_email(
    recipient_email: str,
    meal_plan: dict,
) -> MIMEMultipart:
    msg = MIMEMultipart()
    msg["From"] = EMAIL_SENDER
    msg["To"] = recipient_email
    msg["Subject"] = "Your Weekly Meal Plan"

    body = format_meal_plan_email(meal_plan)
    msg.attach(MIMEText(body, "plain"))

    return msg

## 7. Validation

In [27]:
from pydantic import ValidationError

def run_basic_validation(meal_plan_result: dict) -> dict:
    """Validate the meal plan using the Pydantic models."""
    if not meal_plan_result.get("success"):
        return {
            "passed": False,
            "errors": [meal_plan_result.get("error")],
        }

    try:
        MealPlan.model_validate(meal_plan_result["meal_plan"])
        return {
            "passed": True,
            "errors": [],
        }

    except ValidationError as error:
        errors = [
            f"{'.'.join(str(part) for part in issue['loc'])}: "
            f"{issue['msg']}"
            for issue in error.errors()
        ]

        return {
            "passed": False,
            "errors": errors,
        }

    # Check 1: 7 recipes
    expected_recipe_count = len(DAYS)

    if len(meal_plan["recipes"]) != expected_recipe_count:
        errors.append(
            f"Expected {expected_recipe_count} recipes, "
            f"got {len(meal_plan['recipes'])}"
    )

    for recipe in meal_plan["recipes"]:
        ingredient_count = len(recipe["ingredients"])

    if ingredient_count != INGREDIENTS_PER_RECIPE:
        errors.append(
            f"{recipe['day']}: {ingredient_count} ingredients "
            f"(expected {INGREDIENTS_PER_RECIPE})"
        )

    return {"passed": len(errors) == 0, "errors": errors}

## 8. Gradio Interface

In [24]:
import gradio as gr
import time as pytime

def generate_meal_plan(dietary_pref, theme_requests, additional_prefs):
    return generator.generate(
        dietary_preference=dietary_pref,
        theme_requests=(theme_requests or "").strip() or "random",
        additional_preferences=(additional_prefs or "").strip() or "none",
    )

def format_shopping_list(meal_plan: dict) -> str:
    """Format shopping list section."""
    lines = ["## Shopping List\n"]
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} **{item['name']}**")
    return "\n".join(lines)


def format_single_recipe(recipe: dict) -> str:
    """Format a single recipe compactly."""
    ingredients = format_ingredients(recipe["ingredients"])

    return (
        f"**{recipe['day']}** | {recipe['theme']} | "
        f"*{recipe['dish_name']}*\n"
        f"{ingredients}\n"
)


def generate_plan_streaming(dietary_pref, theme_requests, additional_prefs):
    """Generate meal plan with streaming output."""
    output = "Generating your meal plan..."
    yield output, meal_plan

    result = generate_meal_plan(
    dietary_pref,
    theme_requests,
    additional_prefs,
)

    if not result["success"]:
        yield f"Error: {result['error']}"
        return

    meal_plan = result["meal_plan"]

    # Show shopping list first
    output = f"## Weekly {meal_plan['dietary_preference'].title()} Meal Plan\n\n"
    output += format_shopping_list(meal_plan) + "\n\n---\n\n"
    output += "## Recipes\n\n"
    yield output
    pytime.sleep(0.1)

    # Stream recipes one by one
    for recipe in meal_plan['recipes']:
        output += format_single_recipe(recipe) + "\n"
        yield output
        pytime.sleep(0.05)

    # Validation status
    validation = run_basic_validation(result)
    status = "All checks passed" if validation["passed"] else f"Issues: {', '.join(validation['errors'])}"
    output += f"---\n*Generated in {result['generation_time']}s | {status}*"
    yield output


def send_email_ui(email, meal_plan):
    if not email or "@" not in email:
        return "Enter a valid email address."

    if not meal_plan:
        return "Generate a meal plan before emailing it."

    email_result = send_meal_plan_email(email, meal_plan)
    return email_result["message"]


# Create Gradio interface
with gr.Blocks(title="Weekly Meal Planner") as app:
    gr.Markdown("# Weekly Meal Planner\nGenerate 7 themed dinners with a consolidated shopping list.")

    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(
                choices=["omnivore", "vegetarian", "vegan", "keto"],
                value="omnivore",
                label="Diet"
            )
            theme_input = gr.Textbox(label="Themes (optional)", placeholder="Italian, Mexican...", lines=1)
            additional_input = gr.Textbox(label="Preferences (optional)", placeholder="no nuts, quick meals...", lines=1)
            generate_btn = gr.Button("Generate", variant="primary")

            gr.Markdown("---")
            email_input = gr.Textbox(label="Email", placeholder="you@example.com")
            send_btn = gr.Button("Email Plan")
            email_status = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            output_display = gr.Markdown("Click **Generate** to start.")

    meal_plan_state = gr.State(None)

    generate_btn.click(
        fn=generate_plan_streaming,
        inputs=[dietary_dropdown, theme_input, additional_input],
        outputs=[output_display, meal_plan_state],
)

    send_btn.click(
        fn=send_email_ui,
        inputs=[email_input, meal_plan_state],
        outputs=[email_status],
)

## 9. Launch Application

In [29]:
# Launch the Gradio app
app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://bd07eb6a6569fb5452.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 10. Run Test Cases

Runs every test case through the generator and the basic validation checks. This makes 4 real API calls.

In [30]:
TEST_CASES = [
    {"dietary_preference": "omnivore", "theme_requests": "random", "additional_preferences": "none"},
    {"dietary_preference": "vegetarian", "theme_requests": "Italian, Asian, Mexican", "additional_preferences": "no mushrooms"},
    {"dietary_preference": "vegan", "theme_requests": "random", "additional_preferences": "high protein"},
    {"dietary_preference": "keto", "theme_requests": "Mediterranean, American", "additional_preferences": "none"}
]

for case in TEST_CASES:
    result = generator.generate(
        dietary_preference=case["dietary_preference"],
        theme_requests=case.get("theme_requests", "random"),
        additional_preferences=case.get("additional_preferences", "none")
    )
    validation = run_basic_validation(result)
    print(case["dietary_preference"], "->", "PASS" if validation["passed"] else validation["errors"])

omnivore -> PASS
vegetarian -> PASS
vegan -> PASS
keto -> PASS
